# d03 · 2026-11-06（周五）· 失败路径：让 loop 在模型乱来时也不崩

> 今天的命题只有一个：**工具层要能接住模型的坏输出**，而不是靠模型每次都乖。
>
> 今天**不花一分钱**：所有输入都用自己写的 FakeClient 喂进去，不调 DeepSeek。
> 这恰好是今天要学的东西——失败路径本来就测不到，因为真实模型很少给你失败。

## 时间盒（45 分钟，教室可做）

| 时段 | 干什么 | 判定做到哪就停 |
|---|---|---|
| 0–8 min | 打开 `src/agent/loop.py` 的 `run()`，只读，找「顺利情况下才成立」的行 | §1 表格填满 3 行就停，不改代码 |
| 8–12 min | 读 §2 的三条约束，自己在纸上/这里写 FakeClient 的返回形状 | 能说出 `response.choices[0].message.tool_calls[i]` 上要用到哪 3 个属性 |
| 12–32 min | 改 `run()` 签名 + 写 `invoke_tool()` 兜底 + 写 FakeClient | 四条输入都不抛异常，各打印一行结果 |
| 32–42 min | 跑 §3 的四条输入，填 §4 观测表 | 表格 4 行有真实打印内容 |
| 42–45 min | 填 §5 判定 + §6 坑 | 3 个勾选能勾上 |

## 1. 崩溃点普查（只读自己的代码，别改）

先不看任何外部资料。打开 `src/agent/loop.py`，把 `run()` 里**只在顺利情况下成立**的行找出来。

判据：**模型/工具只要不按你写的剧本走，这里就会抛异常或静默做错事。**

| # | loop.py 行号 | 触发条件（模型或工具怎么做） | 我预判会发生什么（异常类型 / 静默错值） |
|---|---|---|---|
| 1 | 66（原记 84） | 出现不在tools的工具调用 | 会出现报错Keyerr |
| 2 | 70–74（原记 85） | LLM返回的json格式错误，或者json解析参数错误 | ValueErr |
| 3 | 109–113（原记 7-9） | api key或base url 错误 | openai.AuthenticationError |

> 行号已按 2026-11-06 的 170 行版本校正（`TOOL_MAP.get` L66 / `json.loads` L70–74 / client 懒加载 L109–113）；
> 括号里是重构前的位置，保留原记录不改内容。

**预测对照（跑完之后回填，不改上面三行的预测）**

| # | 预测 | 实际 | 判定 |
|---|---|---|---|
| 1 | `KeyError` | 不崩：`TOOL_MAP.get(name)` 返回 `None` 后回灌 `{'ok': False, 'error': 'control_window工具不存在'}` | **预测错** → 进 §6 |
| 2 | `ValueErr`（应为 `ValueError`） | `json.JSONDecodeError` ⊂ `ValueError`，被 L72 接住 | 预测对 |
| 3 | `openai.AuthenticationError` | 未测（今天没跑过错误 key） | 未验证 |

> 只想 3 条，不要多。想完就往下走，**这时候不要动手改代码**。


In [1]:
# 环境自检：确认 kernel 指向项目能用的解释器，且 cwd 找得到 src/
# TRAP: 在 notes/ 里起 kernel 时 cwd 可能是 notes/，别硬写 ../ 或 ../../
import sys, os
from pathlib import Path

print("sys.executable =", sys.executable)
print("cwd =", Path.cwd())
print("这是不是一个 python3.12+：", sys.version_info[:2])

ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path.home() / "code/AgentHarness"]
             if (p / "src" / "agent" / "loop.py").exists()), None)
print("项目根 =", ROOT)
print("loop.py 行数 =", len((ROOT / "src/agent/loop.py").read_text().splitlines()) if ROOT else "未找到")

sys.executable = /opt/homebrew/opt/python@3.15/bin/python3.15
cwd = /Users/wlm/code/AgentHarness/notes
这是不是一个 python3.12+： (3, 15)
项目根 = /Users/wlm/code/AgentHarness
loop.py 行数 = 102


## 2. 三条约束（写代码前先读，读完再动手）

1. **不改工具的业务逻辑。** `control_ac` 的 16–30 上下限今天不许动——报错是今天的观测对象，不是 bug。
2. **不改模块级 `client`。** 让它可注入，不是把它换成假的。判据：改完之后 `python -m src.agent.loop` 走真实 API 还能跑。
3. **兜底不许吞掉信息。** `except` 里必须把「哪个工具 / 哪个参数 / 为什么」带回给模型，只 `print` 或只 `pass` 算没做。

FakeClient 只需要能骗过 `run()`，所以要复现的最小形状：

```
response.choices[0].message.tool_calls   -> None 或 list
response.choices[0].message.tool_calls[i].id            -> str
response.choices[0].message.tool_calls[i].function.name -> str
response.choices[0].message.tool_calls[i].function.arguments -> str（一段 JSON 文本）
response.choices[0].message.content      -> str 或 None
```

> 想清楚一件事：`messages.append(response.choices[0].message)` 这一行意味着 FakeClient 返回的对象**不能是普通 dict**，它得能原样进 `messages` 再被下一次请求接受。

In [ ]:
# FakeClient：只为了让 run() 走到我要的那几条分支
# TODO(you): 用 types.SimpleNamespace 拼出 shape
#   - 第一轮返回一个 tool_call（可指定工具名与 arguments 文本）
#   - 第二轮返回 content="……"、tool_calls=None，让循环正常结束
import json, types

class FakeClient:
    def __init__(self, scripted_calls):
        self.scripted_calls = scripted_calls   # [(name, arguments_text), ...]
        self.calls = 0
        self.chat = types.SimpleNamespace(completions=types.SimpleNamespace(create=self._create))

    def _create(self, **kwargs):
        pass  # TODO(you): 按 scripted_calls[self.calls] 造 response；calls += 1

## 3. 四条输入（一条一条跑，别一次全跑）

| # | 场景 | 输入（工具名 + 参数） | 我想验证什么 |
|---|---|---|---|
| 1 | 正常 | `control_ac` + `{"temp": 24}` | 我的改动没把正常路径弄坏 |
| 2 | 越界 | `control_ac` + `{"temp": 99}` | 工具返回 ok=false 时，错误文本有没有真的回灌给模型 |
| 3 | 未知工具 | `control_window` + `{"percent": 30}` | 工具不在 `TOOL_MAP` 里时，是崩还是回灌一条错误 |
| 4 | 坏 JSON | `control_ac` + `{"temp": 24` | `json.loads` 抛错时，循环还能不能继续 |

> 每跑一条，先**预测**结果，再跑。预测和实际不一样的那条，就是今天真正的收获，记进 §6。

In [ ]:
# TODO(you): 四条输入逐条跑，每条打印：工具名 / 入参 / 工具返回 / 终态 AC_STATUS["temp"]
# TRAP: 不要用 print 判断成功——必须看 AC_STATUS 这个真实状态

## 4. 观测表（真实打印内容）

> 「模型说」那一列是 `CASES` 剧本里写死的字符串，**不是真模型的反应**——FakeClient 不读 `messages`。
> 所以下面第一张表证明的是「循环不崩、错误被回灌、终态没被污染」，不证明模型行为。真模型行为见第二张表。

命令：`cd /Users/wlm/code/AgentHarness && .venv/bin/python -m src.agent.loop`

| # | 工具名 | 入参 | 工具返回（ok / error 原文） | 循环结束方式 | 终态 `AC_STATUS["temp"]` |
|---|---|---|---|---|---|
| 1 | `control_ac` | `{"temp": 24}` | `{'ok': True, 'data': {'temp': 24, 'status': '空调已设置'}}` | 正常结束（第 2 幕收尾） | **24** |
| 2 | `control_ac` | `{"temp": 99}` | `{'ok': False, 'error': 'temp参数必须在16-30之间'}` | 正常结束（第 2 幕收尾） | **27** |
| 3 | `control_window` | `{"percent": 30}` | `{'ok': False, 'error': '工具 control_window 不存在，可用工具：get_weather, control_ac'}` | 正常结束（第 2 幕收尾） | **27** |
| 4 | `control_ac` | `{"temp": 24`（坏 JSON） | `{'ok': False, 'error': "参数不是合法 JSON: Expecting ',' delimiter: line 1 column 12 (char 11)"}` | 正常结束（第 2 幕收尾） | **27** |

### 真 API 复核（1 次，约 0.005 元）

| 项 | 真实输出 |
|---|---|
| 指令 | 把空调调到 24 度 |
| 模型说 | 空调已调到 **24 度** 啦，温度设置成功 ✅ |
| 工具日志 | `[{'tool': 'control_ac', 'args': '{"temp": 24}', 'res': {'ok': True, 'data': {'temp': 24, 'status': '空调已设置'}}}]` |
| 终态 | `{'temp': 24}` |

### 补跑的一条边界（§3 四条之外，复核时发现）

| 输入 | 结果 |
|---|---|
| `control_ac` + `null`（**合法** JSON，但不是 dict） | **已修**（2026-11-06）。原先崩 `UnboundLocalError: cannot access local variable 'e'`——L77 引用了 `except json.JSONDecodeError as e` 块外的 `e`，Python 3 在 `except` 结束时把这个名字删掉（PEP 3110）。 |

修完补跑四种「合法 JSON 但不是对象」，全部回灌结构化错误、不崩：

```
null        -> {'ok': False, 'error': '参数必须是 JSON 对象，收到 NoneType'}
[1, 2]      -> {'ok': False, 'error': '参数必须是 JSON 对象，收到 list'}
24          -> {'ok': False, 'error': '参数必须是 JSON 对象，收到 int'}
{"temp": 24 -> {'ok': False, 'error': "参数不是合法 JSON: Expecting ',' delimiter: line 1 column 12 (char 11)"}
```

> 文案已按错因分开（2026-11-06）：**JSON 语法坏** → `参数不是合法 JSON: ...`；
> **JSON 合法但顶层不是对象** → `参数必须是 JSON 对象，收到 X`。
> 顺带把两处「日志文案 ≠ 回灌文案」的不一致修掉了——原来 `TOOL_LOG` 记 `control_window工具不存在`，
> 回灌给模型的却是 `工具不存在`（模型不知道是哪个工具）。现在两处共用同一个 `err` 变量，不可能再分叉。


## 5. 判定（四条都能勾才算今天完成）

- [x] 未知工具（第 3 条）**没有**抛 `KeyError`，而是回灌了一条模型能读懂的 `role: "tool"` 错误
- [x] 坏 JSON（第 4 条）**没有**让整个循环死掉
- [x] 越界（第 2 条）跑完后 `print(AC_STATUS)` 里温度**不是 99**（错误没有被静默写回）
- [x] `arguments` 是**合法 JSON 但不是对象**（`null` / `[]` / `123`）时，回灌错误而不是崩
      —— 证据：§4 末尾四条实测输出，均返回 `{'ok': False, 'error': ...}`，无异常。

> 第 3 条勾上是重点：它证明「兜底写在工具层之前」，而不是等模型自觉。
>
> 第 1 条的证据已补在 §4「回灌证据」：messages 第 3 条 `role: "tool"`、`content` 里是 `工具不存在`、`tool_call_id` 对得上 `call_1`。
> 注意 `TOOL_LOG` 不算证据——那是你自己记的日志，不是「回灌进历史」。


## 6. 今天踩的坑 / 意外发现

- 坑 1：python包函数不熟悉，types包理解困难（已解决）
- 坑 2：`invoke_tool` L77 的 `e` 出了作用域 —— `arguments` 是**合法 JSON 但不是对象**（`null` / `[]` / `123`）时崩
  `UnboundLocalError: cannot access local variable 'e' where it is not associated with a value`（作用域这个是因为复制代码时的手误）。
  原先记成「FakeClient 返回值格式有问题」，**归因错了**：FakeClient 返回 `SimpleNamespace` 时 `run()` 照跑不崩；
  崩的是 L77 引用了 `except json.JSONDecodeError as e` 块外的 `e`——Python 3 在 `except` 结束时把这个名字删掉（PEP 3110）。
  修它不需要知道 openai 包的任何东西。
- 坑 3：`messages` 里混着三种形状（`dict` / pydantic 对象 / `SimpleNamespace`），
  `json.dumps(messages)` 直接崩 `TypeError: Object of type SimpleNamespace is not JSON serializable`。
  选了「不改 `run()`」这条路线，账单就落在每个消费者身上（打印、trace、评测各归一化一次）。
- 预测错的那条（最重要）：未知工具我觉得会抛出KeyError，但是可以通过get去避免直接在工具层之前拦截异常
- 自我点评 ：理解agent流程没问题，但是对python包的熟悉度差，需要补充python包的语法。


---

### 明天之前不做

- 不装 pytest / pydantic / cmake（今天的 FakeClient 就是测试）
- 不拆包结构（`llm/client.py` + `agent/loop.py` 是下一天）
- 不落 trace 文件（有余力再说；它是 M2 的事，不是今天的验收项）